# Lab 18 — AI Observability & Evaluation

Monitor, evaluate, and audit AI operations using event tables, query history, and LLM-as-judge.

| Item | Detail |
|---|---|
| **Features** | Event Tables, Query History, LLM-as-Judge evaluation |
| **Exam Domain** | 3.0 Gen AI Governance |
| **You'll learn** | Telemetry capture, quality evaluation, automated auditing |


## Step 1 — Environment Setup

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Query History for Cortex AI

Track all Cortex AI function calls through Snowflake's query history.

In [ ]:
SELECT
    QUERY_TEXT,
    USER_NAME,
    EXECUTION_STATUS,
    TOTAL_ELAPSED_TIME / 1000 AS elapsed_sec,
    START_TIME
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE QUERY_TEXT ILIKE '%CORTEX%'
    AND START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY START_TIME DESC
LIMIT 20;

## Step 3 — Event Tables for Telemetry

Event tables capture structured telemetry from UDFs, procedures, and Snowflake functions.

```sql
-- Set up an event table (ACCOUNTADMIN)
CREATE EVENT TABLE IF NOT EXISTS ai_events;
ALTER ACCOUNT SET EVENT_TABLE = 'GENAI_LEARNING.PUBLIC.AI_EVENTS';
```

Event tables store trace data that can be queried for observability.


## Step 4 — LLM-as-Judge Evaluation

Use one LLM to evaluate the quality of another LLM's output. A powerful pattern for automated QA.

In [ ]:
CREATE OR REPLACE TABLE eval_samples (
    sample_id INT AUTOINCREMENT,
    question TEXT,
    context TEXT,
    generated_answer TEXT
);

INSERT INTO eval_samples (question, context, generated_answer) VALUES
('What is Snowflake Time Travel?', 'Time Travel allows accessing data at any point in the last 90 days for Enterprise edition.', 'Snowflake Time Travel lets you access historical data from the past 90 days on Enterprise edition.'),
('What is the max warehouse size?', 'Warehouse sizes range from X-Small to 6X-Large.', 'The largest warehouse size is 4X-Large.'),
('Who founded Snowflake?', 'Snowflake was founded in 2012 by Benoit Dageville, Thierry Cruanes, and Marcin Zukowski.', 'Snowflake was founded by Benoit Dageville and Thierry Cruanes in 2012.');

In [ ]:
SELECT
    sample_id,
    LEFT(question, 50) AS question,
    LEFT(generated_answer, 80) AS answer,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-haiku-4-5',
        [
            {'role': 'system', 'content': 'You are a strict QA evaluator. Score the answer 1-5 on: accuracy (does it match context?), completeness (does it cover the full answer?). Return JSON: {accuracy: N, completeness: N, explanation: "..."}'},
            {'role': 'user', 'content': 'Context: ' || context || '\nQuestion: ' || question || '\nAnswer: ' || generated_answer}
        ],
        {'response_format': {'type': 'json'}, 'temperature': 0.0}
    ) AS evaluation
FROM eval_samples;

## Step 5 — Observability Dashboard Query

Combine usage, cost, and quality metrics for a complete picture.

In [ ]:
SELECT
    'Total Cortex Queries (7d)' AS metric,
    COUNT(*)::STRING AS value
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE QUERY_TEXT ILIKE '%CORTEX%'
    AND START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
UNION ALL
SELECT 'Avg Elapsed Time (sec)',
    ROUND(AVG(TOTAL_ELAPSED_TIME) / 1000, 2)::STRING
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE QUERY_TEXT ILIKE '%CORTEX%'
    AND START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
UNION ALL
SELECT 'Failed Cortex Queries',
    COUNT(*)::STRING
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE QUERY_TEXT ILIKE '%CORTEX%'
    AND EXECUTION_STATUS = 'FAIL'
    AND START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP());

## Key Takeaways

- **Query History** tracks all Cortex AI calls — filter with `ILIKE '%CORTEX%'`.
- **Event Tables** capture structured telemetry for deep observability.
- **LLM-as-Judge** uses one model to evaluate another — great for automated QA.
- Combine usage, cost, and quality metrics for a complete observability dashboard.
- AI observability is a key topic in Exam Domain 3.0 (Governance).
